# Binary Search Trees

```{contents}
:local:
:depth: 2
```

## The BST Invariant

```{index} binary search tree; BST invariant
```


A **binary search tree** is a binary tree with an ordering rule, often called an **invariant**:

- every value in a node's left subtree is less than the node's value;
- every value in a node's right subtree is greater than the node's value.

This rule must be true at every node, not only at the root.


In [ ]:
using System;

var root = new Node(8);
root.Left = new Node(3);
root.Right = new Node(10);
root.Left.Left = new Node(1);
root.Left.Right = new Node(6);
root.Right.Right = new Node(14);

Console.WriteLine($"valid tree: {IsValidBst(root, long.MinValue, long.MaxValue)}");
root.Left!.Right = new Node(9);
Console.WriteLine($"ancestor violation: {IsValidBst(root, long.MinValue, long.MaxValue)}");
Console.WriteLine($"minimum int: {IsValidBst(new Node(int.MinValue), long.MinValue, long.MaxValue)}");
Console.WriteLine($"maximum int: {IsValidBst(new Node(int.MaxValue), long.MinValue, long.MaxValue)}");

bool IsValidBst(Node? node, long min, long max)
{
    if (node is null)
    {
        return true;
    }

    if (node.Value <= min || node.Value >= max)
    {
        return false;
    }

    return IsValidBst(node.Left, min, node.Value)
        && IsValidBst(node.Right, node.Value, max);
}

class Node
{
    public int Value { get; }
    public Node? Left { get; set; }
    public Node? Right { get; set; }

    public Node(int value) => Value = value;
}


The valid range narrows as the recursion moves down the tree. A right child must be greater than its parent and also remain within any limits inherited from ancestors.


A BST stores a set of unique keys in these examples. A duplicate insertion leaves the tree unchanged. If a business application needs multiple records with the same key, decide whether to store a list of records at each key or use a different comparison key. Sending equal values arbitrarily left or right would conflict with the strict invariant used here.

The rule applies to entire subtrees. In the changed example, 9 is greater than its parent 3, but it is still in the left subtree of 8, where every key must be less than 8. Checking only each node against its immediate children would miss this error.

The validator passes an **exclusive lower bound** and an **exclusive upper bound** down the recursion. For the right child of 3 inside the left subtree of 8, the permitted range is `3 < key < 8`. Both ancestor constraints survive. Bounds use `long` because every `int` value, including `int.MinValue` and `int.MaxValue`, must fit strictly inside the initial range. Using those `int` limits as exclusive sentinels would wrongly reject valid boundary keys.

This validator assumes the references already form a tree. It checks ordering, not cycles or shared-node ownership. Validation visits all nodes in a valid tree, so its worst-case work is Θ(n).

## Search

```{index} binary search tree; search
```


BST search uses comparison to discard one subtree at each step. If the target is smaller than the current node, search left. If it is larger, search right. If it is equal, the value has been found.


In [ ]:
using System;

var tree = new BinarySearchTree();
foreach (int value in new[] { 8, 3, 10, 1, 6, 14, 4, 7, 13 })
{
    tree.Insert(value);
}

Console.WriteLine(tree.Contains(7));
Console.WriteLine(tree.Contains(2));

class BinarySearchTree
{
    private Node? root;

    public void Insert(int value)
    {
        root = Insert(root, value);
    }

    public bool Contains(int value)
    {
        Node? current = root;
        while (current is not null)
        {
            if (value == current.Value) return true;
            current = value < current.Value ? current.Left : current.Right;
        }

        return false;
    }

    private static Node Insert(Node? node, int value)
    {
        if (node is null) return new Node(value);
        if (value < node.Value) node.Left = Insert(node.Left, value);
        else if (value > node.Value) node.Right = Insert(node.Right, value);
        return node;
    }

    private sealed class Node
    {
        public int Value { get; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }

        public Node(int value) => Value = value;
    }
}


This `Contains` method is iterative: it walks one path from the root down. A recursive version would follow the same comparisons.


For the tree built above, search for 7 follows `8 → 3 → 6 → 7`. Search for 2 follows `8 → 3 → 1 → null`. The failed search stops when it reaches an empty position; it does not inspect 6, 10, or 14.

At each step, the invariant proves that a discarded subtree cannot contain the target. If 7 is smaller than 8, every key in 8's right subtree is too large. The same argument applies again at the next node. A search examines at most h + 1 nodes in a nonempty tree of height h, giving O(h + 1) time and O(1) extra space for this iterative method.

That bound describes the actual shape. It becomes O(log n) when height is logarithmic, and O(n) for a chain. The basic BST implementation does not promise logarithmic height.

## Insertion

```{index} binary search tree; insertion
```


Insertion searches for the place where the value belongs. When the search reaches an empty child reference, the new node is attached there.

This implementation ignores duplicate values. Another design could store a count at each key. Placing equal keys on one side would require changing the invariant and validator together.


In [ ]:
using System;
using System.Collections.Generic;

var tree = new BinarySearchTree();
foreach (int value in new[] { 8, 3, 10, 1, 6, 14, 4, 7, 13 })
{
    tree.Insert(value);
}

Console.WriteLine(string.Join(", ", tree.InOrder()));

class BinarySearchTree
{
    private Node? root;

    public void Insert(int value)
    {
        root = Insert(root, value);
    }

    public List<int> InOrder()
    {
        var values = new List<int>();
        InOrder(root, values);
        return values;
    }

    private static Node Insert(Node? node, int value)
    {
        if (node is null) return new Node(value);
        if (value < node.Value) node.Left = Insert(node.Left, value);
        else if (value > node.Value) node.Right = Insert(node.Right, value);
        return node;
    }

    private static void InOrder(Node? node, List<int> values)
    {
        if (node is null) return;
        InOrder(node.Left, values);
        values.Add(node.Value);
        InOrder(node.Right, values);
    }

    private sealed class Node
    {
        public int Value { get; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }

        public Node(int value) => Value = value;
    }
}


For a BST, inorder traversal visits values in sorted order. That makes it a useful check when testing insertion.


Recursive insertion returns the root of the updated subtree. The assignments `node.Left = Insert(...)` and `node.Right = Insert(...)` reconnect that result to its parent. The public method similarly assigns the returned root to `root`; otherwise the first insertion into an empty tree would be lost.

Trace adding 4 to a tree containing `8, 3, 10, 1, 6`. Comparisons choose 8's left child, then 3's right child, then 6's left child. The null call creates node 4. Returning from that call assigns it to `6.Left`; the calls at 3 and 8 return their existing nodes. Only one new reference changes. Keys already in the tree keep their relative order.

| Position | Comparison | Action |
| --- | --- | --- |
| 8 | 4 < 8 | Continue left. |
| 3 | 4 > 3 | Continue right. |
| 6 | 4 < 6 | Continue left. |
| null | Empty subtree | Return a new node containing 4. |

Adding 6 again follows the path to the existing 6 and returns it unchanged. Successful and duplicate insertions both cost O(h + 1); recursive insertion uses O(h + 1) call-stack space. A stored Count should increase only when a new node is created.

Sorted inorder output checks the ordering rule, but also compare the output with the intended set. A broken insertion that drops an entire subtree can still produce sorted output. Correctness requires both ordering and preservation of the expected keys.

## Shape and Performance

```{index} binary search tree; height; balanced tree
```


A BST is efficient when its height stays small. If inserted values arrive in sorted order, the tree can become a chain. Then search behaves like scanning a linked list.


In [ ]:
using System;

var balancedLike = new BinarySearchTree();
foreach (int value in new[] { 8, 3, 10, 1, 6, 14, 4, 7, 13 })
{
    balancedLike.Insert(value);
}

var chain = new BinarySearchTree();
foreach (int value in new[] { 1, 3, 4, 6, 7, 8, 10, 13, 14 })
{
    chain.Insert(value);
}

Console.WriteLine($"Mixed insertion height: {balancedLike.Height()}");
Console.WriteLine($"Sorted insertion height: {chain.Height()}");

class BinarySearchTree
{
    private Node? root;

    public void Insert(int value) => root = Insert(root, value);
    public int Height() => Height(root);

    private static Node Insert(Node? node, int value)
    {
        if (node is null) return new Node(value);
        if (value < node.Value) node.Left = Insert(node.Left, value);
        else if (value > node.Value) node.Right = Insert(node.Right, value);
        return node;
    }

    private static int Height(Node? node)
    {
        if (node is null) return -1;
        return 1 + Math.Max(Height(node.Left), Height(node.Right));
    }

    private sealed class Node
    {
        public int Value { get; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }
        public Node(int value) => Value = value;
    }
}


Balanced tree structures, such as AVL trees and red-black trees, add rules that keep height under control. This chapter focuses on the basic BST first so the ordering idea is clear.


A perfect binary tree of height h contains `1 + 2 + ... + 2^h = 2^(h + 1) - 1` nodes. Each extra level can roughly double the number of stored keys, so a tree with well-distributed branches can have logarithmic height. A chain adds only one node per level. Inserting n distinct sorted keys into this basic implementation performs roughly `0 + 1 + ... + (n - 1)` comparisons, giving Θ(n²) total construction work.

| Operation | Basic BST cost | Reason |
| --- | --- | --- |
| Search, add, remove, find minimum | O(h + 1) | Follow one or a constant number of root-to-descendant paths. |
| Inorder traversal, uncached height, full validation | Θ(n) in the worst case | Examine every node. |
| Read stored Count | O(1) | Read a maintained field. |

Random-looking input is not a guarantee of balanced height. AVL and red-black trees enforce additional rules after updates; this chapter leaves their rotations for later study. Choose a balanced library structure when a service needs a logarithmic worst-case guarantee. For the simple teaching tree, the input order is part of the performance story.

## Removal and Subtree Replacement

```{index} binary search tree; removal; inorder successor
```

Removing from a BST has more cases than search or insertion:

| Case | Basic idea |
|---|---|
| Leaf node | Remove the parent's reference to it. |
| One child | Replace the node with its child. |
| Two children | Replace the node's value with its inorder successor or predecessor, then remove that moved value. |

The two-child case is why removal is often taught after search and insertion. It is not conceptually impossible, but it has more structural bookkeeping.


In [ ]:
using System;

var root = new Node(8);
root.Left = new Node(3);
root.Right = new Node(10);
root.Right.Right = new Node(14);
root.Right.Right.Left = new Node(13);

Console.WriteLine($"Minimum: {Minimum(root)}");
Console.WriteLine($"Maximum: {Maximum(root)}");

int Minimum(Node node)
{
    Node current = node;
    while (current.Left is not null)
    {
        current = current.Left;
    }

    return current.Value;
}

int Maximum(Node node)
{
    Node current = node;
    while (current.Right is not null)
    {
        current = current.Right;
    }

    return current.Value;
}

class Node
{
    public int Value { get; }
    public Node? Left { get; set; }
    public Node? Right { get; set; }

    public Node(int value) => Value = value;
}


Finding the minimum and maximum helps explain removal: an inorder successor is the smallest value in the right subtree.


Use `TicketTree` below as an ordered set of active service-ticket IDs. `Add` and `Remove` report whether membership changed; `Count` changes once per successful operation. `TryMinimum` distinguishes an empty tree from a valid key zero. The public API keeps nodes private so callers cannot silently break the invariant.

Deletion returns the updated subtree root, just as insertion does. A leaf returns null. A node with one child returns that child. These cases also handle deleting the root because the public method stores the return value in `root`.

For two children, find the smallest key in the right subtree, the **inorder successor**. Copy that key into the current node, then delete the original successor from the right subtree. The successor has no left child; it may have a right child, which must be reattached. Replacing the key without removing its original node would leave a duplicate. Removing the successor by discarding its entire subtree would lose records.

This class stores IDs only. For nodes containing an ID and a record, transfer the matching record as well as the ID, or move the entire node. Keeping an old record with a new key would corrupt the application even if the key ordering remained valid.

In [ ]:
using System;
using System.Collections.Generic;

var tickets = new TicketTree();
foreach (int id in new[] { 8, 3, 12, 1, 6, 10, 14, 11 }) tickets.Add(id);
Console.WriteLine($"duplicate added={tickets.Add(8)}, count={tickets.Count}");
foreach (int id in new[] { 1, 10, 8, 99 })
{
    Console.WriteLine($"remove {id}: {tickets.Remove(id)}");
    Console.WriteLine($"keys={string.Join(", ", tickets.InOrder())}; count={tickets.Count}; valid={tickets.IsValid()}");
}

public class TicketTree
{
    private Node? root;
    public int Count { get; private set; }
    public int Height => NodeHeight(root);

    public bool Add(int id)
    {
        bool added = false;
        root = Insert(root, id, ref added);
        if (added) Count++;
        return added;
    }

    public bool Contains(int id)
    {
        Node? current = root;
        while (current is not null)
        {
            if (id == current.Id) return true;
            current = id < current.Id ? current.Left : current.Right;
        }
        return false;
    }

    public bool Remove(int id)
    {
        bool removed = false;
        root = Delete(root, id, ref removed);
        if (removed) Count--;
        return removed;
    }

    public bool TryMinimum(out int id)
    {
        if (root is null) { id = default; return false; }
        id = Minimum(root).Id;
        return true;
    }

    public List<int> InOrder()
    {
        var ids = new List<int>();
        Visit(root, ids);
        return ids;
    }

    public bool IsValid() => Validate(root, long.MinValue, long.MaxValue);

    private static Node Insert(Node? node, int id, ref bool added)
    {
        if (node is null) { added = true; return new Node(id); }
        if (id < node.Id) node.Left = Insert(node.Left, id, ref added);
        else if (id > node.Id) node.Right = Insert(node.Right, id, ref added);
        return node;
    }

    private static Node? Delete(Node? node, int id, ref bool removed)
    {
        if (node is null) return null;
        if (id < node.Id) node.Left = Delete(node.Left, id, ref removed);
        else if (id > node.Id) node.Right = Delete(node.Right, id, ref removed);
        else
        {
            removed = true;
            if (node.Left is null) return node.Right;
            if (node.Right is null) return node.Left;
            Node successor = Minimum(node.Right);
            node.Id = successor.Id;
            bool successorRemoved = false;
            node.Right = Delete(node.Right, successor.Id, ref successorRemoved);
        }
        return node;
    }

    private static Node Minimum(Node node)
    {
        while (node.Left is not null) node = node.Left;
        return node;
    }

    private static void Visit(Node? node, List<int> ids)
    {
        if (node is null) return;
        Visit(node.Left, ids);
        ids.Add(node.Id);
        Visit(node.Right, ids);
    }

    private static int NodeHeight(Node? node) => node is null ? -1
        : 1 + Math.Max(NodeHeight(node.Left), NodeHeight(node.Right));

    private static bool Validate(Node? node, long lower, long upper)
    {
        if (node is null) return true;
        return lower < node.Id && node.Id < upper
            && Validate(node.Left, lower, node.Id)
            && Validate(node.Right, node.Id, upper);
    }

    private class Node
    {
        public int Id { get; set; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }
        public Node(int id) => Id = id;
    }
}

The original ticket tree and the final tree after closing 1, 10, and 8 show how the references change:

```text
        8                       11
       / \                     /  \
      3   12                  3    12
     / \  / \                  \     \
    1  6 10 14                  6    14
          \
          11
```

Read the demonstration as three different reference updates:

| Remove | Case at that moment | Reference update |
| --- | --- | --- |
| 1 | Leaf under 3 | Set `3.Left` to null. |
| 10 | One child, 11 | Set `12.Left` to node 11. |
| 8 | Root with two children | Replace root ID with 11, then set `12.Left` to null. |
| 99 | Missing key | Return false; preserve keys and Count. |

For root 8, the successor is 11 after the earlier removal of 10. It is the smallest key greater than 8. Every old left key remains less than 11, and every remaining right key is greater than 11. Deleting the old successor location restores uniqueness.

The outer removal sets `removed = true`. The internal successor deletion uses a separate flag, and only the public method decrements Count. One requested deletion removes one logical key even though the implementation also visits the successor location. The sum of these paths is O(h + 1), not a full traversal.

**Predict:** rebuild the original demonstration tree and remove 8 first. Its successor is 10, which has right child 11. After replacement, `12.Left` must refer to 11. Then test removal from an empty tree, a singleton, and a root with only one child. These cases reveal missed root assignments and lost child links.

## Exercise

```{index} binary search tree; exercise
```


In [ ]:
// Exercise: Complete search
// Fill in Contains so it returns true for 7 and false for 2.

using System;

var tree = new BinarySearchTree();
foreach (int value in new[] { 8, 3, 10, 1, 6, 14, 4, 7, 13 })
{
    tree.Insert(value);
}

Console.WriteLine(tree.Contains(7));
Console.WriteLine(tree.Contains(2));

class BinarySearchTree
{
    private Node? root;

    public void Insert(int value)
    {
        root = Insert(root, value);
    }

    public bool Contains(int value)
    {
        Node? current = root;
        while (current is not null)
        {
            // Your code starts here.
            throw new NotImplementedException("Complete Contains before running.");

            // Your code ends here.
        }

        return false;
    }

    private static Node Insert(Node? node, int value)
    {
        if (node is null) return new Node(value);
        if (value < node.Value) node.Left = Insert(node.Left, value);
        else if (value > node.Value) node.Right = Insert(node.Right, value);
        return node;
    }

    private sealed class Node
    {
        public int Value { get; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }
        public Node(int value) => Value = value;
    }
}


In [1]:
// Solution
using System;

var tree = new BinarySearchTree();
foreach (int value in new[] { 8, 3, 10, 1, 6, 14, 4, 7, 13 })
{
    tree.Insert(value);
}

Console.WriteLine(tree.Contains(7));
Console.WriteLine(tree.Contains(2));

class BinarySearchTree
{
    private Node? root;

    public void Insert(int value)
    {
        root = Insert(root, value);
    }

    public bool Contains(int value)
    {
        Node? current = root;
        while (current is not null)
        {
            if (value == current.Value)
            {
                return true;
            }

            current = value < current.Value ? current.Left : current.Right;
        }

        return false;
    }

    private static Node Insert(Node? node, int value)
    {
        if (node is null) return new Node(value);
        if (value < node.Value) node.Left = Insert(node.Left, value);
        else if (value > node.Value) node.Right = Insert(node.Right, value);
        return node;
    }

    private sealed class Node
    {
        public int Value { get; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }
        public Node(int value) => Value = value;
    }
}


True
False


## Summary

```{index} Summary
```


A BST adds an ordering invariant to a binary tree. Search and insertion follow one path when the tree is well shaped, but sorted insertion can produce a tall chain.

```{rubric} Footnotes
```

[^1]: Self-balancing BSTs keep height small by rotating nodes after insertions or removals. They are a later data-structures topic.
